# Sentinel-2 tree segmentation 

U-Net on Satlas's 9-band Sentinel-2 backbone, trained on the Forest class of Copernicus-Bench DFC2020
Torchgeo downloads the dataset (8.2 GB) on the first run. Needs torchgeo

In [1]:
from pathlib import Path

DATA = Path("./datasets/dfc2020")
SAVE = Path("./local_satlas_tree_sentinel_ms")
BANDS = ["B04", "B03", "B02", "B05", "B06", "B07", "B08", "B11", "B12"]  # order Satlas MS expects
BATCH_SIZE, EPOCHS = 16, 10

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchgeo.datasets import CopernicusBenchDFC2020S2

FOREST = CopernicusBenchDFC2020S2.classes.index("Forest")


def to_tree_sample(sample):
    image = sample["image"]
    image[:3] = (image[:3] / 3000).clamp(0, 1)
    image[3:] = (image[3:] / 8160).clamp(0, 1)
    return {"image": image, "mask": (sample["mask"] == FOREST).long()}


def make_loader(split):
    dataset = CopernicusBenchDFC2020S2(DATA, split=split, bands=BANDS, transforms=to_tree_sample, download=True)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=split == "train", num_workers=4)


train_loader, val_loader = make_loader("train"), make_loader("val")
(DATA / "dfc2020.zip").unlink(missing_ok=True)  # already extracted frees the size

In [ ]:
from lightning.pytorch import Trainer
from torchgeo.models import ResNet50_Weights
from torchgeo.trainers import SemanticSegmentationTask

# unet
task = SemanticSegmentationTask(model="unet", backbone="resnet50", weights=ResNet50_Weights.SENTINEL2_SI_MS_SATLAS,
                                in_channels=len(BANDS), task="binary", loss="bce", lr=1e-4)

Trainer(max_epochs=EPOCHS, default_root_dir=SAVE).fit(
    task, train_dataloaders=train_loader, val_dataloaders=val_loader)

SAVE.mkdir(parents=True, exist_ok=True)
torch.save(task.model.state_dict(), SAVE / "satlas_ms_unet_tree.pt")